# 제1장: AI 거버넌스 개념과 필요성

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch01.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

### AI 거버넌스의 개념과 필요성

## AI 거버넌스의 정의와 범위(Scope)

### AI 거버넌스란 무엇인가

### 데이터 거버넌스, IT 거버넌스, AI 거버넌스의 관계

### 거버넌스의 진화: IT에서 데이터로, 데이터에서 AI로

### AI 거버넌스의 구성 요소: 5대 핵심 영역

### AI 거버넌스의 범위: 무엇이 포함되는가

## AI 거버넌스가 필요한 이유

### 비즈니스 관점: 신뢰, 리스크, 가치

### 기술적 관점: 품질, 재현성, 유지보수성

### 규제적 관점: 준수 요구사항 충족

### 세 관점의 통합: 지속가능한 AI 실현

## AI 거버넌스 부재의 결과: 현장의 기록

### 사례 C: 의료 AI 플랫폼의 기술 거버넌스 공백

### 사례 B: 금융기관 AI 시스템의 데이터 거버넌스 실패

### 사례 A: 공공기관 생성형 AI 프로젝트의 조직적 붕괴

### 글로벌 AI 거버넌스 실패: 알고리즘 뒤에 숨겨진 인간의 고통

### 세 사례의 교훈과 한국 기업에의 시사점

### 거버넌스 Health Check 자동화

**AI 거버넌스 Health Check 자동화 도구**

In [ ]:
"""
AI Governance Health Check Tool
- 프로젝트의 거버넌스 준수 상태를 자동 진단
- 5대 영역(정책, 프로세스, 기술, 조직, 문화) 점검
"""
import os
import json
from dataclasses import dataclass, field
from typing import List, Dict
from enum import Enum

class Severity(Enum):
    CRITICAL = "CRITICAL"  # 즉시 조치 필요
    WARNING = "WARNING"    # 개선 권고
    INFO = "INFO"          # 참고 사항

@dataclass
class CheckResult:
    category: str          # 거버넌스 영역
    check_name: str        # 점검 항목명
    passed: bool           # 통과 여부
    severity: Severity     # 심각도
    message: str           # 상세 메시지
    recommendation: str    # 개선 권고사항

@dataclass
class GovernanceReport:
    project_name: str
    results: List[CheckResult] = field(default_factory=list)

    @property
    def score(self) -> float:
        if not self.results:
            return 0.0
        passed = sum(1 for r in self.results if r.passed)
        return round(passed / len(self.results) * 100, 1)

    @property
    def critical_issues(self) -> List[CheckResult]:
        return [r for r in self.results
                if not r.passed and r.severity == Severity.CRITICAL]

    def summary(self) -> Dict:
        return {
            "project": self.project_name,
            "total_checks": len(self.results),
            "passed": sum(1 for r in self.results if r.passed),
            "failed": sum(1 for r in self.results if not r.passed),
            "score": self.score,
            "critical_count": len(self.critical_issues),
        }

class GovernanceHealthCheck:
    """AI 프로젝트 거버넌스 Health Check 수행 클래스"""

    def __init__(self, project_root: str):
        self.project_root = project_root
        self.report = GovernanceReport(
            project_name=os.path.basename(project_root)
        )

    # --- 정책(Policy) 영역 점검 ---
    def check_policy_documents(self):
        """필수 정책 문서 존재 여부 점검"""
        required_docs = {
            "MODEL_CARD.md": "모델 카드",
            "ETHICS_REVIEW.md": "윤리 검토서",
            "RISK_ASSESSMENT.md": "위험 평가서",
            "DATA_GOVERNANCE.md": "데이터 거버넌스 정책",
        }
        for filename, doc_name in required_docs.items():
            exists = os.path.exists(
                os.path.join(self.project_root, "docs", filename)
            )
            self.report.results.append(CheckResult(
                category="정책",
                check_name=f"{doc_name} 존재 여부",
                passed=exists,
                severity=Severity.CRITICAL,
                message=f"{doc_name} {'존재' if exists else '미존재'}",
                recommendation="" if exists else
                    f"docs/{filename} 작성 필요"
            ))

    # --- 프로세스(Process) 영역 점검 ---
    def check_cicd_pipeline(self):
        """CI/CD 파이프라인 설정 존재 여부 점검"""
        ci_files = [
            ".github/workflows", ".gitlab-ci.yml",
            "Jenkinsfile", ".circleci/config.yml"
        ]
        found = any(
            os.path.exists(os.path.join(self.project_root, f))
            for f in ci_files
        )
        self.report.results.append(CheckResult(
            category="프로세스",
            check_name="CI/CD 파이프라인 설정",
            passed=found,
            severity=Severity.CRITICAL,
            message="CI/CD 파이프라인 " +
                    ("설정됨" if found else "미설정"),
            recommendation="" if found else
                "자동화된 빌드/테스트/배포 파이프라인 구축 필요"
        ))

    def check_test_coverage(self):
        """테스트 코드 존재 여부 점검"""
        test_dirs = ["tests", "test", "spec"]
        found = any(
            os.path.isdir(os.path.join(self.project_root, d))
            for d in test_dirs
        )
        self.report.results.append(CheckResult(
            category="프로세스",
            check_name="테스트 코드 디렉토리",
            passed=found,
            severity=Severity.CRITICAL,
            message="테스트 디렉토리 " +
                    ("존재" if found else "미존재"),
            recommendation="" if found else
                "단위/통합/공정성 테스트 작성 필요"
        ))

    # --- 기술(Technology) 영역 점검 ---
    def check_model_versioning(self):
        """모델 버전 관리 체계 점검"""
        version_indicators = [
            "mlflow", "wandb", "dvc", "mlflow.log_model"
        ]
        # requirements.txt 또는 pyproject.toml 검사
        req_file = os.path.join(
            self.project_root, "requirements.txt"
        )
        found = False
        if os.path.exists(req_file):
            with open(req_file, "r") as f:
                content = f.read().lower()
                found = any(v in content
                           for v in version_indicators)
        self.report.results.append(CheckResult(
            category="기술",
            check_name="모델 버전 관리 도구",
            passed=found,
            severity=Severity.WARNING,
            message="모델 버전 관리 도구 " +
                    ("감지됨" if found else "미감지"),
            recommendation="" if found else
                "MLflow, DVC 등 모델 레지스트리 도입 권장"
        ))

    def check_monitoring_config(self):
        """모니터링 설정 존재 여부 점검"""
        monitoring_indicators = [
            "evidently", "whylabs", "arize",
            "prometheus", "grafana", "monitoring"
        ]
        found = any(
            os.path.exists(
                os.path.join(self.project_root, f"monitoring")
            )
            for _ in [1]
        )
        self.report.results.append(CheckResult(
            category="기술",
            check_name="모니터링 설정",
            passed=found,
            severity=Severity.WARNING,
            message="모니터링 설정 " +
                    ("존재" if found else "미존재"),
            recommendation="" if found else
                "모델 성능/드리프트 모니터링 체계 구축 권장"
        ))

    # --- 전체 점검 실행 ---
    def run_all_checks(self) -> GovernanceReport:
        """모든 거버넌스 점검 항목 실행"""
        self.check_policy_documents()
        self.check_cicd_pipeline()
        self.check_test_coverage()
        self.check_model_versioning()
        self.check_monitoring_config()
        return self.report

    def print_report(self):
        """거버넌스 점검 결과 출력"""
        report = self.run_all_checks()
        summary = report.summary()

        print("=" * 60)
        print(f"AI Governance Health Check: {summary['project']}")
        print(f"Score: {summary['score']}% "
              f"({summary['passed']}/{summary['total_checks']})")
        print("=" * 60)

        for result in report.results:
            status = "PASS" if result.passed else "FAIL"
            icon = "[+]" if result.passed else "[-]"
            print(f"  {icon} [{result.category}] "
                  f"{result.check_name}: {status}")
            if not result.passed:
                print(f"      -> {result.recommendation}")

        if report.critical_issues:
            print(f"\n!! {len(report.critical_issues)} "
                  f"CRITICAL issues found !!")

# 사용 예시
if __name__ == "__main__":
    checker = GovernanceHealthCheck("/path/to/ai-project")
    checker.print_report()

## AI 거버넌스 성숙도 모델

### 성숙도 모델의 필요성

### 5단계 성숙도 프레임워크

### 구성 요소별 성숙도 특성 매트릭스

### 성숙도 단계별 규제 준수 매핑

### 성숙도 진단 방법론

### 성숙도 향상을 위한 단계별 권장 활동

### 제1장 요약